# Lab 2 | Titanic data analysis

## Step 1: Setting up the Project

In [ ]:
"""
Titanic Data Analysis and JSON Export
Author: Carlos Martinez Boto
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""

import pandas as pd
import numpy as np
import json
from pathlib import Path
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

## Step 2: Importing and Exploring the Data

In [ ]:
# Import the dataset into a DataFrame, understand the data
df = pd.read_csv('data/titanic.csv')

In [ ]:
# Verification
print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

## Step 3: Calculating Descriptive Statistics

Calculate mean, median, and standard deviation for numeric columns

In [ ]:
numeric_columns = df.select_dtypes(include=[np.number]) # Identify all numeric columns in the dataset
desc_stat = numeric_columns.agg(['mean', 'median', 'std']) # Calculate mean, median, and standard deviation for each numeric column
print(desc_stat)

# Step 4: Identifying Missing Values

Count and analyze missing values in the dataset

In [ ]:
# Count missing values
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)

missing_count = df.isna().sum()
missing_percent = (missing_count / len(df)) * 100
print(missing_percent)

## Step 5: Feature Engineering

Create new features that might help differentiate between survivors and non-survivors

In [ ]:
# Create a copy of the dataframe for feature engineering
df_features = df.copy()

In [ ]:
# Feature 1: Family Size: SibSp (Siblings/Spouses) + Parch (Parents/Children) + 1 (themselves)
df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))

In [ ]:
# Feature 2: Is Alone
df_features['IsAlone'] = df_features['FamilySize'] == 1
print(df_features[['FamilySize', 'IsAlone']].head(10))

In [ ]:
# Feature 3: Age Groups
def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Minors'
    elif age < 30:
        return 'Young adults'
    elif age < 50:
        return 'Adults'
    else:
        return 'Older adults'
df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))

In [ ]:
# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50)
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)
 
# Here is an example to get you started:
print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)
 
# Statistical test: Do these features help differentiate?
print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)
 
survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]
 
print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

## Step 6: Creating a Data Export Class

Create a Python class to structure and export data to JSON format

In [38]:
print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [ ]:


class Passenger:
    """
    Represents a passenger with all their information.
    """
    def __init__(self, passenger_id, name, age, sex, survived, pclass, 
                 fare, embarked=None, family_size=None, is_alone=None, title=None):
        # Tip: Use pd.notna() to check if a value is not null/NaN
        # Tip: Convert to appropriate types (int, float, str)
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name         = str(name) if pd.notna(name) else None
        self.age          = float(age) if pd.notna(age) else None
        self.sex          = str(sex) if pd.notna(sex) else None
        self.survived     = int(survived) if pd.notna(survived) else None
        self.pclass       = int(pclass) if pd.notna(pclass) else None
        self.fare         = float(fare) if pd.notna(fare) else None
        self.embarked     = str(embarked) if pd.notna(embarked) else None
        self.family_size  = int(family_size) if pd.notna(family_size) else None
        self.is_alone     = bool(is_alone) if pd.notna(is_alone) else None
        self.title        = str(title) if pd.notna(title) else None
        pass
    
    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""
        # Tip: Dictionary keys should match the attribute names
        return {
            'passenger_id': self.passenger_id,
            'name': self.name,
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked,
            'family_size': self.family_size,
            'is_alone': self.is_alone,
            'title': self.title
        }


In [ ]:
class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []  # Will store Passenger objects
        self._create_passengers()
    
    def _create_passengers(self):
        """Create Passenger objects from dataframe."""
        # Tip: Use self.dataframe.iterrows() to loop through rows
        # Tip: Use row.get('ColumnName', default_value) to safely get values
        for idx, row in self.dataframe.iterrows():
            # TODO: Create a Passenger object with data from the row
            new_psg = Passenger(passenger_id, name, age, sex, survived, pclass, 
                 fare, embarked=None, family_size=None, is_alone=None, title=None)
            # Example of getting PassengerId:
            # passenger_id = row.get('PassengerId', idx)
            
            # TODO: Create the Passenger object and append to self.passengers
            self.passengers.append(new_psg)
    
    def to_json(self, filename='titanic_data.json'):
        """Export dataset to JSON file."""
        # TODO: Create a dictionary with metadata and passenger data
        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                # TODO: Add more metadata fields
                # Tip: Calculate total_passengers from self.passengers
                # Tip: Calculate survival_rate using self.dataframe['Survived'].mean()
            },
            'passengers': []  # TODO: Convert all passenger objects to dictionaries
            # Tip: Use list comprehension with p.to_dict() for each passenger
        }
        
        # TODO: Write the data to a JSON file
        # Tip: Use json.dump() with indent=2 for readable formatting
        
        print(f"Data exported to {filename}")
        return data
    
    def get_summary_stats(self):
        """Get summary statistics."""
        # TODO: Calculate and return summary statistics
        # Tip: Use list comprehensions to filter and calculate
        # Example for counting survived passengers:
        # survived_count = sum(1 for p in self.passengers if p.survived == 1)
        
        return {
            'total_passengers': None,  # TODO: Calculate total passengers
            'survived': None,  # TODO: Count passengers who survived
            'did_not_survive': None,  # TODO: Count passengers who didn't survive
            # TODO: Add more statistics (average_age, average_fare)
            # Tip: Remember to handle None values when calculating averages
        }


In [ ]:
# TODO: Create dataset object and export
# Check if df_engineered exists and is not empty
if 'df_engineered' in locals() and not df_engineered.empty:
    # TODO: Create a TitanicDataset object
    # dataset = TitanicDataset(...)
    
    # TODO: Print basic information about the dataset
    
    # TODO: Get and display summary statistics
    # Tip: Call get_summary_stats() and iterate through the results
    
    # TODO: Export to JSON (optional - uncomment when ready)
    # dataset.to_json('titanic_data.json')
    
    pass  # Remove this when you add your code